# Persiapan Dataset YOLO 18K - Multi-Class

Notebook ini menyiapkan dataset dalam format YOLO untuk training model multi-class
**Plate Count Reader** dengan 4 kelas: colony (0), bubble (1), dust (2), crack (3).

## Parameter Utama (sesuai train_18k_gpu1.py)
- **Output**: `data/yolo_18k_multiclass/`
- **PL_CONF**: 0.20 (pseudo-label confidence)
- **MAX_PER_BUCKET**: 1000
- **Synthetic**: 1000 bubble, 800 dust, 500 crack
- **Split**: 80/12/8 (train/val/test)
- **MIN_COLONY_AREA**: 8
- **Seed**: 42

## Konfigurasi

In [ ]:
# ============================================================
# KONFIGURASI UTAMA (SESUAI train_18k_gpu1.py)
# ============================================================
import os
from pathlib import Path

# Path
WORKSPACE = Path('/media/lambda_one/DFSSD04/project/healtcare')
AGAR = WORKSPACE / 'data' / 'agar'
DATA_DIR = AGAR / 'dataset'

# Nested structure (SESUAI train_18k_gpu1.py)
U2NET_DIR = DATA_DIR / 'dataset_for_u2net' / 'dataset_for_u2net'
RESNET_DIR = DATA_DIR / 'dataset_for_resnet' / 'dataset_for_resnet'
MODEL_PATH = WORKSPACE / 'models' / 'best_plate_count_reader.pt'

# Output dataset YOLO (SESUAI train_18k_gpu1.py)
YOLO_DIR = WORKSPACE / 'data' / 'yolo_18k_multiclass'

# Class names
CLASS_NAMES = ['colony', 'bubble', 'dust', 'crack']
NUM_CLASSES = len(CLASS_NAMES)

# Pseudo-labeling config (SESUAI train_18k_gpu1.py)
MAX_PSEUDO_PER_BUCKET = 1000  # Max images per count bucket
PL_CONF = 0.20                # Confidence threshold (was 0.25, FIXED)

# Synthetic generation config (SESUAI train_18k_gpu1.py)
NUM_SYNTHETIC_BUBBLE = 1000   # was 500, FIXED
NUM_SYNTHETIC_DUST = 800      # was 500, FIXED
NUM_SYNTHETIC_CRACK = 500     # was 300, FIXED

# Split ratios (SESUAI train_18k_gpu1.py: 80/12/8)
TRAIN_RATIO = 0.80
VAL_RATIO = 0.12
TEST_RATIO = 0.08

# Training parameters
MIN_COLONY_AREA = 8
MAX_COLONIES = 3000
IMG_SIZE = 640
SEED = 42

print('Konfigurasi dimuat (SESUAI train_18k_gpu1.py)')
print(f'   Workspace  : {WORKSPACE}')
print(f'   YOLO output: {YOLO_DIR}')
print(f'   Classes    : {CLASS_NAMES}')
print(f'   PL_CONF    : {PL_CONF}')
print(f'   Synthetic  : {NUM_SYNTHETIC_BUBBLE} bubble, {NUM_SYNTHETIC_DUST} dust, {NUM_SYNTHETIC_CRACK} crack')
print(f'   Split      : {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}')

In [ ]:
# Import libraries
import glob
import json
import random
import shutil
from collections import Counter, defaultdict

import cv2
import numpy as np
from scipy import ndimage
from PIL import Image, ImageDraw

random.seed(SEED)
np.random.seed(SEED)

print('Libraries berhasil diimport')

## Step 1: Konversi Mask U2Net ke YOLO Boxes

Konversi mask segmentasi biner (255=colony, 0=background) menjadi format YOLO bbox.
Struktur: `dataset_for_u2net/dataset_for_u2net/{train,valid}_mask/colony_detecting/`

In [ ]:
def mask_to_yolo_boxes(mask_path, class_id=0, min_area=MIN_COLONY_AREA):
    """
    Konversi mask biner ke format YOLO bounding boxes.
    SESUAI train_18k_gpu1.py logic.
    
    Format YOLO: <class_id> <x_center> <y_center> <width> <height>
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    
    h, w = mask.shape
    binary = (mask > 127).astype(np.uint8)
    labeled, num_features = ndimage.label(binary)
    
    if num_features > MAX_COLONIES:
        return []  # Skip images with too many colonies
    
    yolo_lines = []
    for label_id in range(1, num_features + 1):
        comp = labeled == label_id
        if comp.sum() < min_area:
            continue
        
        ys, xs = np.where(comp)
        x_min, x_max = int(xs.min()), int(xs.max())
        y_min, y_max = int(ys.min()), int(ys.max())
        
        cx = max(0, min(1, (x_min + x_max) / 2.0 / w))
        cy = max(0, min(1, (y_min + y_max) / 2.0 / h))
        bw = max(0.001, min(1, (x_max - x_min) / w))
        bh = max(0.001, min(1, (y_max - y_min) / h))
        
        yolo_lines.append(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    
    return yolo_lines

# Proses semua mask U2Net (SESUAI train_18k_gpu1.py)
u2net_yolo_data = []
processed = 0
skipped = 0

for split_name in ['train', 'valid']:
    mask_dir = U2NET_DIR / f'{split_name}_mask' / 'colony_detecting'
    img_dir = U2NET_DIR / split_name
    
    if not mask_dir.exists():
        print(f'Mask dir tidak ditemukan: {mask_dir}')
        continue
    
    mask_files = sorted([f for f in mask_dir.iterdir() if f.name.endswith('_mask.png')])
    print(f'{split_name}: {len(mask_files)} mask files')
    
    for mask_path in mask_files:
        img_name = mask_path.name.replace('_mask.png', '.png')
        img_path = img_dir / img_name
        
        if not img_path.exists():
            skipped += 1
            continue
        
        yolo_lines = mask_to_yolo_boxes(mask_path, class_id=0)
        if yolo_lines:
            u2net_yolo_data.append((img_path, '\n'.join(yolo_lines)))
            processed += 1

print(f'Konversi U2Net selesai:')
print(f'   Diproses : {processed} gambar')
print(f'   Dilewati : {skipped}')

## Step 2: Pseudo-Labeling AGAR ResNet

SESUAI train_18k_gpu1.py: iterate `for split in ['train', 'val']` dan sampling maksimal 1000 per bucket.
Skip bucket count=0 (koloni=0 tidak ada koloni untuk dilabeli).

In [ ]:
# Load model YOLOv8 yang sudah ada
from ultralytics import YOLO

if MODEL_PATH.exists():
    model = YOLO(str(MODEL_PATH))
    print(f'Model loaded: {MODEL_PATH}')
    print(f'   Classes: {model.names}')
else:
    print(f'Model tidak ditemukan: {MODEL_PATH}')
    print('   Pseudo-labeling akan di-skip')

In [ ]:
# Pseudo-labeling dataset ResNet (SESUAI train_18k_gpu1.py)
# FIXED: iterate both 'train' and 'val' splits, skip count=0
pseudo_labeled_data = []
count_per_bucket = defaultdict(int)

if MODEL_PATH.exists():
    all_images = []
    for split in ['train', 'val']:  # FIXED: was only 'train'
        split_dir = RESNET_DIR / split
        if not split_dir.exists():
            continue
        for cd in sorted(os.listdir(split_dir)):
            try:
                cnt = int(cd)
            except ValueError:
                continue
            if cnt == 0:  # Skip count=0 (no colonies)
                continue
            cp = split_dir / cd
            if not cp.exists():
                continue
            for f in sorted(os.listdir(cp)):
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    all_images.append((cp / f, cnt))
    
    print(f'ResNet images (count>0): {len(all_images)}')
    
    # Sampling per bucket
    random.seed(SEED)
    by_count = defaultdict(list)
    for p, c in all_images:
        by_count[c].append(p)
    
    sampled = []
    for c, imgs in sorted(by_count.items()):
        random.shuffle(imgs)
        sampled.extend(imgs[:MAX_PSEUDO_PER_BUCKET])
        print(f'  Bucket {c}: {min(len(imgs), MAX_PSEUDO_PER_BUCKET)} gambar dipilih (total: {len(imgs)})')
    
    random.shuffle(sampled)
    print(f'\nTotal sampled for pseudo-labeling: {len(sampled)}')
    
    # Batch inference
    labeled = 0
    bs = 32
    for i in range(0, len(sampled), bs):
        batch = sampled[i:i + bs]
        results = model(batch, conf=PL_CONF, verbose=False, imgsz=IMG_SIZE, device='1')
        
        for j, result in enumerate(results):
            if result.boxes is None or len(result.boxes) == 0:
                continue
            yolo_lines = []
            for k in range(len(result.boxes)):
                conf_val = float(result.boxes.conf[k])
                if conf_val < PL_CONF:
                    continue
                xywhn = result.boxes.xywhn[k].cpu().numpy()
                vals = [max(0, min(1, float(v))) for v in xywhn]
                cx, cy, w, h = vals
                w = max(0.001, w)
                h = max(0.001, h)
                yolo_lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
            
            if yolo_lines:
                pseudo_labeled_data.append((batch[j], '\n'.join(yolo_lines)))
                labeled += 1
        
        if (i // bs) % 10 == 0:
            print(f'  Progress: {i}/{len(sampled)}, labeled: {labeled}')
    
    print(f'\nPseudo-labeling selesai: {labeled} gambar')
else:
    print('Model tidak tersedia, skip pseudo-labeling')

## Step 3: Generasi Artefak Sintetis

SESUAI train_18k_gpu1.py: 1000 bubble, 800 dust, 500 crack.
Synthetic artifacts menambah label ke gambar yang sudah ada (bukan membuat gambar baru).

In [ ]:
# Generate synthetic artifacts (SESUAI train_18k_gpu1.py logic)
# Menambah label sintetis ke gambar yang sudah ada
synthetic_data = []

# Ambil gambar dari raw data (U2Net + pseudo-labeled)
all_base = list(u2net_yolo_data) + list(pseudo_labeled_data)
print(f'Base images untuk synthetic: {len(all_base)}')

random.seed(SEED)
random.shuffle(all_base)

# Bubble: class 1
print(f'Generating {NUM_SYNTHETIC_BUBBLE} bubble annotations...')
for i in range(NUM_SYNTHETIC_BUBBLE):
    img_path, existing_labels = all_base[i % len(all_base)]
    new_lines = existing_labels.strip().split('\n')
    for _ in range(random.randint(1, 3)):
        cx = random.uniform(0.1, 0.9)
        cy = random.uniform(0.1, 0.9)
        w = random.uniform(0.02, 0.08)
        h = random.uniform(0.02, 0.08)
        new_lines.append(f'1 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    synthetic_data.append((img_path, '\n'.join(new_lines), f'aug_bubble_{i:04d}'))

bubble_count = len(synthetic_data)
print(f'   {bubble_count} bubble samples')

# Dust: class 2
print(f'Generating {NUM_SYNTHETIC_DUST} dust annotations...')
for i in range(NUM_SYNTHETIC_DUST):
    img_path, existing_labels = all_base[(i + 100) % len(all_base)]
    new_lines = existing_labels.strip().split('\n')
    for _ in range(random.randint(3, 8)):
        cx = random.uniform(0.05, 0.95)
        cy = random.uniform(0.05, 0.95)
        w = random.uniform(0.003, 0.015)
        h = random.uniform(0.003, 0.015)
        new_lines.append(f'2 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    synthetic_data.append((img_path, '\n'.join(new_lines), f'aug_dust_{i:04d}'))

dust_count = len(synthetic_data) - bubble_count
print(f'   {dust_count} dust samples')

# Crack: class 3
print(f'Generating {NUM_SYNTHETIC_CRACK} crack annotations...')
for i in range(NUM_SYNTHETIC_CRACK):
    img_path, existing_labels = all_base[(i + 200) % len(all_base)]
    new_lines = existing_labels.strip().split('\n')
    for _ in range(random.randint(1, 2)):
        cx, cy = random.uniform(0.2, 0.8), random.uniform(0.2, 0.8)
        if random.random() > 0.5:
            w, h = random.uniform(0.1, 0.3), random.uniform(0.005, 0.02)
        else:
            w, h = random.uniform(0.005, 0.02), random.uniform(0.1, 0.3)
        new_lines.append(f'3 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    synthetic_data.append((img_path, '\n'.join(new_lines), f'aug_crack_{i:04d}'))

crack_count = len(synthetic_data) - bubble_count - dust_count
print(f'   {crack_count} crack samples')
print(f'\nTotal synthetic samples: {len(synthetic_data)}')

## Step 4: Split Dataset

Split dataset secara stratified: 80% train, 12% val, 8% test (SESUAI train_18k_gpu1.py).

In [ ]:
# Gabungkan semua data
all_data = []

# U2Net data (semua colony)
for img_path, label_content in u2net_yolo_data:
    num_labels = len(label_content.strip().split('\n'))
    all_data.append(('u2net', img_path, label_content, num_labels))

# Pseudo-labeled data
for img_path, label_content in pseudo_labeled_data:
    num_labels = len(label_content.strip().split('\n'))
    all_data.append(('pseudo', img_path, label_content, num_labels))

# Synthetic data
for img_path, label_content, name in synthetic_data:
    lines = label_content.strip().split('\n')
    class_ids = [int(line.split()[0]) for line in lines if line.strip()]
    dominant = Counter(class_ids).most_common(1)[0][0] if class_ids else 0
    all_data.append(('synthetic', img_path, label_content, dominant))

print(f'Total data sebelum split: {len(all_data)}')
print(f'   U2Net      : {sum(1 for d in all_data if d[0] == "u2net")}')
print(f'   Pseudo     : {sum(1 for d in all_data if d[0] == "pseudo")}')
print(f'   Synthetic  : {sum(1 for d in all_data if d[0] == "synthetic")}')

In [ ]:
# Stratified split (SESUAI train_18k_gpu1.py)
random.seed(SEED)

by_class = defaultdict(list)
for item in all_data:
    source, img_path, label, strat_key = item
    by_class[strat_key].append(item)

train_data, val_data, test_data = [], [], []

for c, group in sorted(by_class.items()):
    random.shuffle(group)
    n = len(group)
    nt = max(1, int(n * TRAIN_RATIO))
    nv = max(1, int(n * VAL_RATIO))
    train_data.extend(group[:nt])
    val_data.extend(group[nt:nt + nv])
    test_data.extend(group[nt + nv:])

print(f'Split selesai:')
print(f'   Train: {len(train_data)} ({len(train_data)/len(all_data)*100:.1f}%)')
print(f'   Val  : {len(val_data)} ({len(val_data)/len(all_data)*100:.1f}%)')
print(f'   Test : {len(test_data)} ({len(test_data)/len(all_data)*100:.1f}%)')

In [ ]:
# Copy files ke struktur YOLO dataset (SESUAI train_18k_gpu1.py)
# Format: {YOLO_DIR}/{split}/images/ dan {YOLO_DIR}/{split}/labels/
for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    img_dir = YOLO_DIR / split_name / 'images'
    lbl_dir = YOLO_DIR / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    for idx, (source, img_path, label_content, strat_key) in enumerate(split_data):
        stem = Path(img_path).stem
        suffix = Path(img_path).suffix or '.jpg'
        base_name = f'{source}_{idx:06d}'
        
        dst_img = img_dir / f'{base_name}{suffix}'
        dst_lbl = lbl_dir / f'{base_name}.txt'
        
        if not dst_img.exists():
            try:
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                    cv2.imwrite(str(dst_img), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
                else:
                    continue
            except Exception:
                continue
        
        with open(str(dst_lbl), 'w') as f:
            f.write(label_content)
    
    n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    n_lbl = len(list(lbl_dir.glob('*.txt')))
    print(f'{split_name}: {n_img} images, {n_lbl} labels')

## Step 5: Generate data.yaml

In [ ]:
# Generate data.yaml (SESUAI train_18k_gpu1.py format)
data_yaml_content = f"""path: {YOLO_DIR}
train: train/images
val: val/images
test: test/images

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""

yaml_path = YOLO_DIR / 'data.yaml'
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(str(yaml_path), 'w') as f:
    f.write(data_yaml_content)

print(f'data.yaml disimpan di: {yaml_path}')
print()
print(data_yaml_content)

## Verifikasi Dataset

In [ ]:
# Verifikasi dataset
print('=' * 60)
print('VERIFIKASI DATASET YOLO 18K MULTICLASS')
print('=' * 60)

for split in ['train', 'val', 'test']:
    img_dir = YOLO_DIR / split / 'images'
    lbl_dir = YOLO_DIR / split / 'labels'
    
    n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    n_lbl = len(list(lbl_dir.glob('*.txt')))
    
    class_counts = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(str(lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1
    
    print(f'\n{split.upper()}:')
    print(f'   Images: {n_img}, Labels: {n_lbl}')
    print(f'   Match : {"OK" if n_img == n_lbl else "MISMATCH!"}')
    
    total_boxes = sum(class_counts.values())
    for cls_id in range(NUM_CLASSES):
        pct = (class_counts.get(cls_id, 0) / total_boxes * 100) if total_boxes > 0 else 0
        print(f'   {CLASS_NAMES[cls_id]:<10}: {class_counts.get(cls_id, 0):>6} boxes ({pct:.1f}%)')

print(f'\n{"=" * 60}')

## Ringkasan

In [ ]:
# Ringkasan
print('=' * 70)
print('RINGKASAN PERSIAPAN DATASET YOLO 18K MULTI-CLASS')
print('=' * 70)
print()
print(f'{"Parameter":<35} {"Nilai"}')
print('-' * 70)
print(f'{"Lokasi dataset":<35} {YOLO_DIR}')
print(f'{"Jumlah kelas":<35} {NUM_CLASSES} ({CLASS_NAMES})')
print(f'{"Ukuran gambar":<35} {IMG_SIZE}x{IMG_SIZE}')
print(f'{"PL confidence":<35} {PL_CONF}')
print(f'{"Synthetic bubble/dust/crack":<35} {NUM_SYNTHETIC_BUBBLE}/{NUM_SYNTHETIC_DUST}/{NUM_SYNTHETIC_CRACK}')
print(f'{"Split ratio":<35} {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}')
print(f'{"MIN_COLONY_AREA":<35} {MIN_COLONY_AREA}')
print(f'{"Seed":<35} {SEED}')
print('=' * 70)